In [ ]:
import pandas as pd

page_list = all_reviews

reviews_data = []

for i, page in enumerate(page_list):

    # remove unicode chars
    rev = page["review"].replace('"',"").encode('ascii', 'ignore').decode('ascii')


    words = rev.split()

    # ignore short reviews
    if len(words) < 30: 
        continue 

    author_vote = 1 if page['voted_up'] == True else 0
    other_votes = page['votes_up']
    weighted_vote_score = page["weighted_vote_score"]
    
    reviews_data.append({
        'review': rev,
        'author_vote': author_vote,
        'other_votes': other_votes,
        'weighted_vote_score': weighted_vote_score
    })

# Create DataFrame
df = pd.DataFrame(reviews_data)
print(f"Created DataFrame with {len(df)} reviews")

# Save to CSV
df.to_csv(f'reviews_{appid}.csv', index=False)

In [ ]:
# remove tmp folder
import shutil
shutil.rmtree('tmp')

In [ ]:
from flask import Flask, render_template, request
import requests
from textblob import TextBlob

app = Flask(__name__)

# --------------------------
# Steam Review Scraper
# --------------------------

def get_reviews(appid, num_reviews=100):

    url = (
        f"https://store.steampowered.com/appreviews/"
        f"{appid}?json=1&num_per_page={num_reviews}"
    )

    response = requests.get(url)
    data = response.json()

    reviews = []

    for review in data["reviews"]:
        reviews.append(review["review"])

    return reviews


# --------------------------
# Sentiment Analysis
# --------------------------

def get_sentiment(text):

    polarity = TextBlob(text).sentiment.polarity

    if polarity > 0.1:
        return "Positive"

    elif polarity < -0.1:
        return "Negative"

    return "Neutral"


# --------------------------
# Categorization
# --------------------------

usability_keywords = [
    "bug",
    "crash",
    "lag",
    "fps",
    "server",
    "performance"
]

mechanic_keywords = [
    "movement",
    "weapon",
    "perk",
    "battle pass",
    "matchmaking"
]

comparison_keywords = [
    "better than",
    "worse than",
    "compared to",
    "similar to"
]


def categorize_review(review):

    review_lower = review.lower()

    categories = []

    if any(word in review_lower for word in usability_keywords):
        categories.append("Usability Issues")

    if any(word in review_lower for word in mechanic_keywords):
        categories.append("New Mechanic Reception")

    if any(word in review_lower for word in comparison_keywords):
        categories.append("Competitive Analysis")

    if len(categories) == 0:
        categories.append("Game Reception")

    return categories


# --------------------------
# Routes
# --------------------------

@app.route("/", methods=["GET", "POST"])
def home():

    results = []

    if request.method == "POST":

        appid = request.form["appid"]

        reviews = get_reviews(appid)

        for review in reviews:

            sentiment = get_sentiment(review)
            categories = categorize_review(review)

            results.append({
                "review": review[:300],
                "sentiment": sentiment,
                "categories": ", ".join(categories)
            })

    return render_template(
        "index.html",
        results=results
    )


if __name__ == "__main__":
    app.run(debug=True)


In [ ]:
#HTML webpage app
<!DOCTYPE html>
<html>

<head>
<title>Steam Review Analyzer</title>
</head>

<body>

    <h1>Steam Review Analyzer</h1>

    <form method="POST">

        <label>Steam App ID:</label>

        <input
            type="text"
            name="appid"
            placeholder="730 for CS2"
            required>

        <button type="submit">
            Analyze Reviews
        </button>

    </form>

    <hr>

    {% for item in results %}

    <h3>{{ item.sentiment }}</h3>

    <strong>
        {{ item.categories }}
    </strong>

    <p>
    {{ item.review }}
    </p>

    <hr>

    {% endfor %}

</body>
</html>
